# Task 4: Commenter-Level Graph Analysis

**Project:** Coordinated Amplification and Misinformation Detection in Global YouTube Conflict Narratives  
**Course:** CS 418 (UIC)

This notebook builds a **commenter co-occurrence graph** from YouTube comments on Russia-Ukraine conflict videos:

1. **Bot-like accounts** via composite behavioral scoring
2. **Communities** of coordinated commenters via Louvain clustering
3. **Influential commenters** via PageRank

**Data engineering highlight:** All heavy aggregation (commenter stats, edge-list computation) is pushed server-side to BigQuery, avoiding the need to pull 1M+ raw comment rows into local memory.

## 1. Setup & Data Loading

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from utils.bq_helpers import load_comments, load_videos, COLORS

print('Libraries loaded successfully')

Libraries loaded successfully


In [2]:
# Load all comments from local CSV
print('Loading comments from local CSV...')
comments = load_comments()
print(f'Comments loaded: {comments.shape}')

# Compute commenter stats locally (equivalent to the BigQuery aggregation)
print('Computing commenter stats...')
commenter_stats = comments.dropna(subset=['author_channel_id']).groupby('author_channel_id').agg(
    video_count=('video_id', 'nunique'),
    comment_count=('comment_id', 'count'),
    first_comment=('published_at', 'min'),
    last_comment=('published_at', 'max'),
).reset_index()

# Compute active_days
comments_with_date = comments.dropna(subset=['author_channel_id']).copy()
comments_with_date['date'] = comments_with_date['published_at'].dt.date
active_days = comments_with_date.groupby('author_channel_id')['date'].nunique().reset_index()
active_days.columns = ['author_channel_id', 'active_days']
commenter_stats = commenter_stats.merge(active_days, on='author_channel_id', how='left')

print(f'Commenter stats: {commenter_stats.shape}')
print(f'\nDescriptive stats:')
commenter_stats[['video_count', 'comment_count', 'active_days']].describe()

Loading comments from local CSV...


Comments loaded: (168980, 7)
Computing commenter stats...
Commenter stats: (122101, 6)

Descriptive stats:


,video_count,comment_count,active_days
count,122101.000000,122101.000000,122101.000000
mean,1.304568,1.383936,1.277639
std,1.069293,1.388453,0.995469
min,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000
max,27.000000,93.000000,30.000000


## 2. Server-Side Co-Occurrence Edge List

### Why this matters for Data Engineering interviews

Computing commenter co-occurrence naively would require:
1. Pulling 1M+ comment rows into Python
2. Building an in-memory self-join (O(n^2) memory)

Instead, we **push the computation to BigQuery** — the warehouse handles the self-join on its distributed engine, and we only pull back the aggregated edge list. This is a fundamental data engineering pattern: **query pushdown**.

In [3]:
# Compute co-occurrence edge list locally
# (In production, this would be a server-side BigQuery self-join for efficiency)
print('Computing co-occurrence edge list locally...')
print('Building commenter-video pairs...')

# Get distinct (author_channel_id, video_id) pairs
commenter_videos = comments.dropna(subset=['author_channel_id'])[
    ['author_channel_id', 'video_id']
].drop_duplicates()

print(f'Unique commenter-video pairs: {len(commenter_videos):,}')

# Self-join on video_id to find commenters who co-occur
# For efficiency, only keep commenters with >= 2 videos
active_commenters = commenter_videos.groupby('author_channel_id').filter(
    lambda x: len(x) >= 2
)
print(f'Active commenter-video pairs (>=2 videos): {len(active_commenters):,}')

# Merge on video_id to get co-occurrences
print('Computing co-occurrences (self-join)...')
merged = active_commenters.merge(active_commenters, on='video_id', suffixes=('_a', '_b'))
merged = merged[merged['author_channel_id_a'] < merged['author_channel_id_b']]

# Count shared videos per pair
edge_list = merged.groupby(
    ['author_channel_id_a', 'author_channel_id_b']
).size().reset_index(name='shared_videos')

# Filter to pairs sharing >= 3 videos
edge_list = edge_list[edge_list['shared_videos'] >= 3].reset_index(drop=True)
edge_list.columns = ['commenter_a', 'commenter_b', 'shared_videos']

print(f'\nEdge list: {edge_list.shape}')
print(f'Shared videos stats:')
print(edge_list['shared_videos'].describe())

Computing co-occurrence edge list locally...
Building commenter-video pairs...


Unique commenter-video pairs: 159,289


Active commenter-video pairs (>=2 videos): 56,194
Computing co-occurrences (self-join)...



Edge list: (49835, 3)
Shared videos stats:
count    49835.000000
mean         3.810876
std          1.531285
min          3.000000
25%          3.000000
50%          3.000000
75%          4.000000
max         20.000000
Name: shared_videos, dtype: float64


## 3. Network Construction

In [4]:
# Build NetworkX graph from edge list
G = nx.Graph()

for _, row in tqdm(edge_list.iterrows(), total=len(edge_list), desc='Building graph'):
    G.add_edge(row['commenter_a'], row['commenter_b'], weight=row['shared_videos'])

# Add node attributes from commenter stats
stats_dict = commenter_stats.set_index('author_channel_id').to_dict('index')
for node in G.nodes():
    if node in stats_dict:
        for key, val in stats_dict[node].items():
            G.nodes[node][key] = val

print(f'Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'Density: {nx.density(G):.6f}')
print(f'Connected components: {nx.number_connected_components(G)}')

# Largest connected component
largest_cc = max(nx.connected_components(G), key=len)
print(f'Largest connected component: {len(largest_cc):,} nodes')

Building graph:   0%|          | 0/49835 [00:00<?, ?it/s]

Building graph:  17%|█▋        | 8247/49835 [00:00<00:00, 82462.61it/s]

Building graph:  35%|███▍      | 17195/49835 [00:00<00:00, 86584.67it/s]

Building graph:  53%|█████▎    | 26165/49835 [00:00<00:00, 88005.91it/s]

Building graph:  70%|███████   | 35105/49835 [00:00<00:00, 88553.46it/s]

Building graph:  88%|████████▊ | 44064/49835 [00:00<00:00, 88924.96it/s]

Building graph: 100%|██████████| 49835/49835 [00:00<00:00, 88305.15it/s]

Graph: 6,008 nodes, 49,835 edges
Density: 0.002762
Connected components: 153
Largest connected component: 2,006 nodes


## 4. Bot Detection

Composite bot score based on behavioral heuristics:
- **High volume score** (0.3): Percentile rank of video_count
- **Burst score** (0.3): comment_count / active_days > 20
- **Cross-channel score** (0.2): video_count / active_days > 5
- **Network score** (0.2): Degree centrality (highly connected commenters)

In [5]:
# Get degree centrality for nodes in graph
degree_cent = nx.degree_centrality(G)

# Build bot scoring DataFrame
bot_df = commenter_stats.copy()
bot_df['active_days'] = bot_df['active_days'].clip(lower=1)  # avoid division by zero

# High volume score (normalized percentile rank of video_count)
bot_df['high_volume_score'] = bot_df['video_count'].rank(pct=True) * 0.3

# Burst score (comments per active day)
bot_df['comments_per_day'] = bot_df['comment_count'] / bot_df['active_days']
bot_df['burst_score'] = np.clip(bot_df['comments_per_day'] / 20, 0, 1) * 0.3

# Cross-channel score (videos per active day)
bot_df['videos_per_day'] = bot_df['video_count'] / bot_df['active_days']
bot_df['cross_channel_score'] = np.clip(bot_df['videos_per_day'] / 5, 0, 1) * 0.2

# Network score (degree centrality from commenter graph)
bot_df['degree_centrality'] = bot_df['author_channel_id'].map(degree_cent).fillna(0)
max_dc = bot_df['degree_centrality'].max()
bot_df['network_score'] = (bot_df['degree_centrality'] / max_dc * 0.2) if max_dc > 0 else 0

# Composite bot score
bot_df['bot_score'] = (
    bot_df['high_volume_score'] +
    bot_df['burst_score'] +
    bot_df['cross_channel_score'] +
    bot_df['network_score']
)

bot_df['is_likely_bot'] = (bot_df['bot_score'] >= 0.5).astype(int)

n_bots = bot_df['is_likely_bot'].sum()
print(f'Total commenters: {len(bot_df):,}')
print(f'Likely bots (score >= 0.5): {n_bots:,} ({n_bots/len(bot_df)*100:.1f}%)')
print(f'\nBot score distribution:')
print(bot_df['bot_score'].describe())

Total commenters: 122,101
Likely bots (score >= 0.5): 78 (0.1%)

Bot score distribution:
count    122101.000000
mean          0.207155
std           0.059515
min           0.143486
25%           0.181653
50%           0.181653
75%           0.181653
max           0.824001
Name: bot_score, dtype: float64


In [6]:
# Bot score distribution
fig = px.histogram(bot_df, x='bot_score', nbins=50,
    title='Distribution of Composite Bot Scores',
    color_discrete_sequence=[COLORS['primary']])
fig.add_vline(x=0.5, line_dash='dash', line_color='red',
    annotation_text='Bot threshold (0.5)')
fig.update_layout(height=400, width=900, xaxis_title='Bot Score', yaxis_title='Count')
fig.show()

In [7]:
# Top 20 suspected bots
top_bots = bot_df.nlargest(20, 'bot_score')
print('Top 20 Suspected Bots:')
top_bots[['author_channel_id', 'comment_count', 'video_count', 'active_days',
          'comments_per_day', 'bot_score']].reset_index(drop=True)

Top 20 Suspected Bots:


,author_channel_id,comment_count,video_count,active_days,comments_per_day,bot_score
0,UCYmDgE1CU5_gPvuaVAFnDRQ,20,19,1,20.0,0.824001
1,UCASwUzHCVSYnNSvU1Nq5C2A,13,10,1,13.0,0.696015
2,UC1NF71EwP41VdjAU1iXdLkw,12,12,1,12.0,0.682607
3,UC7WbwVwH9851TySoHfRimGA,11,10,1,11.0,0.678674
4,UCgSObzXfYEdO4UlBas2c2nw,29,2,1,29.0,0.647912
5,UC7zVXgA_tzcfEtGWQr4RVOw,9,8,1,9.0,0.640455
6,UCWm3kIESZozB1slfrJ7-ksA,9,8,1,9.0,0.638556
7,UC_W4p81y6sx0ojOucduhOHg,8,8,1,8.0,0.618493
8,UCwmn5V4NHGFdLzGAiyyVNWQ,7,7,1,7.0,0.609265
9,UCVMLtv8zpAlIVYki3okiVgw,7,5,1,7.0,0.605011


In [8]:
# Activity scatter: comment count vs video count
sample = bot_df.sample(min(10000, len(bot_df)), random_state=42)
fig = px.scatter(sample, x='video_count', y='comment_count',
    color='bot_score', color_continuous_scale='RdYlBu_r',
    title='Commenter Activity (colored by bot score)',
    log_x=True, log_y=True, opacity=0.4,
    labels={'video_count': 'Videos Commented On', 'comment_count': 'Total Comments'})
fig.update_layout(height=550, width=900)
fig.show()

## 5. Community Detection

Louvain clustering on top-5000 commenters by degree for tractable community detection.

In [9]:
# Take subgraph of top commenters by degree
top_n = min(5000, G.number_of_nodes())
top_nodes = sorted(G.nodes(), key=lambda n: G.degree(n), reverse=True)[:top_n]
G_sub = G.subgraph(top_nodes).copy()

print(f'Subgraph for clustering: {G_sub.number_of_nodes():,} nodes, {G_sub.number_of_edges():,} edges')

# Louvain community detection
partition = community_louvain.best_partition(G_sub, weight='weight', random_state=42)
modularity = community_louvain.modularity(partition, G_sub, weight='weight')

n_communities = len(set(partition.values()))
print(f'Communities found: {n_communities}')
print(f'Modularity: {modularity:.4f}')

# Community sizes
comm_sizes = pd.Series(partition).value_counts().reset_index()
comm_sizes.columns = ['cluster_id', 'size']
print(f'\nTop 10 community sizes:')
print(comm_sizes.head(10).to_string())

Subgraph for clustering: 5,000 nodes, 48,699 edges


Communities found: 103
Modularity: 0.9177

Top 10 community sizes:
   cluster_id  size
0           8   343
1          13   295
2           4   239
3           0   236
4          26   204
5          28   196
6           9   193
7          29   192
8          10   183
9           6   171


In [10]:
# Community size bar chart
fig = px.bar(comm_sizes.head(20), x='cluster_id', y='size',
    title='Top 20 Commenter Communities by Size',
    labels={'cluster_id': 'Community', 'size': 'Members'})
fig.update_layout(height=400, width=900)
fig.show()

## 6. Cross-Channel Analysis

Check if commenter clusters map to channel clusters from Task 2.

In [11]:
# Try to load channel clusters from Task 2
try:
    channel_clusters = pd.read_csv('../task2_network_analysis/outputs/channel_clusters.csv')
    print(f'Loaded channel clusters: {channel_clusters.shape}')
    
    # Load video data to get commenter-to-channel mapping
    videos = load_videos(['video_id', 'channel_id'])
    
    # Map comments to channels via video_id
    commenter_channels = comments.dropna(subset=['author_channel_id'])[
        ['author_channel_id', 'video_id']
    ].drop_duplicates().merge(videos[['video_id', 'channel_id']], on='video_id', how='left')
    
    commenter_channels = commenter_channels.dropna(subset=['channel_id']).drop_duplicates()
    print(f'Commenter-channel links: {len(commenter_channels):,}')
    
    # Merge with channel clusters
    commenter_channels = commenter_channels.merge(
        channel_clusters[['channel_id', 'cluster_id']],
        on='channel_id', how='left'
    )
    
    # For each commenter community, what channel clusters do they comment on?
    commenter_channels['commenter_cluster'] = commenter_channels['author_channel_id'].map(partition)
    cross = commenter_channels.dropna(subset=['commenter_cluster', 'cluster_id'])
    if len(cross) > 0:
        cross_tab = pd.crosstab(cross['commenter_cluster'].astype(int),
                                cross['cluster_id'].astype(int), normalize='index')
        print(f'\nCross-tabulation (commenter community vs channel cluster):')
        print(cross_tab.head(10))

except FileNotFoundError:
    print('Channel clusters not found. Run Task 2 notebook first.')
    print('Skipping cross-channel analysis.')

Loaded channel clusters: (263, 7)


Commenter-channel links: 159,289

Cross-tabulation (commenter community vs channel cluster):
cluster_id                0         1         2
commenter_cluster                              
0                  0.000951  0.000000  0.999049
1                  0.000000  0.000000  1.000000
2                  0.000000  0.000000  1.000000
3                  0.000000  1.000000  0.000000
4                  0.729150  0.002383  0.268467
5                  1.000000  0.000000  0.000000
6                  0.020672  0.246770  0.732558
7                  0.019481  0.980519  0.000000
8                  0.998381  0.000540  0.001079
9                  0.986607  0.000000  0.013393


## 7. Top Influencers — PageRank

In [12]:
# PageRank on commenter subgraph
print('Computing PageRank...')
pr = nx.pagerank(G_sub, weight='weight')

# Build leaderboard
leaderboard = pd.DataFrame({
    'author_channel_id': list(pr.keys()),
    'pagerank': list(pr.values()),
}).sort_values('pagerank', ascending=False)

# Merge with stats and bot scores
leaderboard = leaderboard.merge(
    bot_df[['author_channel_id', 'comment_count', 'video_count', 'active_days', 'bot_score']],
    on='author_channel_id', how='left'
)
leaderboard['cluster_id'] = leaderboard['author_channel_id'].map(partition)
leaderboard['degree'] = leaderboard['author_channel_id'].map(lambda n: G_sub.degree(n) if n in G_sub else 0)

print('\nTop 20 Influential Commenters:')
top20 = leaderboard.head(20).reset_index(drop=True)
top20.index += 1
top20[['author_channel_id', 'comment_count', 'video_count', 'pagerank', 'bot_score', 'cluster_id']]

Computing PageRank...

Top 20 Influential Commenters:


,author_channel_id,comment_count,video_count,pagerank,bot_score,cluster_id
1,UC6cF_2V1vNKx-nFdGd8cOcA,20,20,0.003019,0.545212,13
2,UCky-W_xOMThidqxNrwJUsUA,27,27,0.002588,0.475961,6
3,UC0gpekdxUu5gN5SPGd9Nrdg,20,20,0.002516,0.490877,9
4,UC6mDrmm-g1lAO2VjupvkVgA,25,23,0.002471,0.442402,4
5,UC_hL0FBiK0CwYGYXKaGLA0g,70,19,0.002394,0.565915,13
6,UCqWfAHz6V6fja9s0EYi8qwQ,18,16,0.002235,0.481399,0
7,UCxfusAsd_n5AMFheiipyaCw,20,20,0.002140,0.411937,25
8,UCg5cUVRmX59ySHKGuoV0PXA,41,24,0.002120,0.431985,4
9,UCeXWPng7yGBIGfTmMg5cK4w,17,16,0.001916,0.491854,13
10,UCS_0nLB8QsIp1D6fQ-O9eMw,18,17,0.001864,0.427623,100


In [13]:
# PageRank bar chart
fig = px.bar(top20, x='pagerank', y='author_channel_id', orientation='h',
    color='bot_score', color_continuous_scale='RdYlBu_r',
    title='Top 20 Commenters by PageRank (colored by bot score)')
fig.update_layout(height=600, width=1000, yaxis=dict(autorange='reversed'))
fig.show()

## 8. Save Outputs

In [14]:
import os
os.makedirs('outputs', exist_ok=True)

# Save bot scores
bot_output = bot_df[['author_channel_id', 'comment_count', 'video_count',
                      'active_days', 'bot_score', 'is_likely_bot']]
bot_output.to_csv('outputs/commenter_bot_scores.csv', index=False)
print(f'Saved outputs/commenter_bot_scores.csv ({len(bot_output):,} rows)')

# Save commenter clusters
cluster_output = leaderboard[['author_channel_id', 'cluster_id', 'pagerank', 'degree']]
cluster_output.to_csv('outputs/commenter_clusters.csv', index=False)
print(f'Saved outputs/commenter_clusters.csv ({len(cluster_output):,} rows)')

print('\nDone!')

Saved outputs/commenter_bot_scores.csv (122,101 rows)
Saved outputs/commenter_clusters.csv (5,000 rows)

Done!
